## TCGA/mutations exploratory data analysis 

We want to know how many genes/cancer types pass our mutation filters (>5% mutated, >50 samples mutated).

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

import pancancer_evaluation.config as cfg
import pancancer_evaluation.utilities.data_utilities as du

np.random.seed(cfg.default_seed)

First, we load the relevant TCGA preprocessed data, downloaded in the `download_data.ipynb` script.

In [ ]:
print('Loading gene label data...', file=sys.stderr)
genes_df = du.load_top_50()
sample_info_df = du.load_sample_info(verbose=True)

# this returns a tuple of dataframes, unpack it below
pancancer_data = du.load_pancancer_data(verbose=True)
(sample_freeze_df,
 mutation_df,
 copy_loss_df,
 copy_gain_df,
 mut_burden_df) = pancancer_data

rnaseq_df = du.load_expression_data(verbose=True)

# standardize columns of expression dataframe
print('Standardizing columns of expression data...', file=sys.stderr)
rnaseq_df[rnaseq_df.columns] = StandardScaler().fit_transform(rnaseq_df[rnaseq_df.columns])

In [ ]:
print(rnaseq_df.shape)
rnaseq_df.iloc[:5, :5]

In [ ]:
mutation_genes = set(mutation_df.columns)
copy_loss_genes = set(copy_loss_df.columns)
copy_gain_genes = set(copy_gain_df.columns)
overlap_genes = mutation_genes.intersection(copy_loss_genes.intersection(copy_gain_genes))
print('Genes with mutation information: {}'.format(len(overlap_genes)))

In [ ]:
cancer_types_df = pd.read_csv(
    Path(cfg.data_dir, 'tcga_sample_counts.tsv').resolve(),
    sep='\t'
)
print('Total cancer type/mutation combinations: {}'.format(cancer_types_df.shape[0] * len(overlap_genes)))
cancer_types_df.head()

Next, we filter gene/cancer type combinations by the number and proportion of mutations, and put the results in a dataframe.

For now we're just doing this for the top 50 most mutated genes in TCGA (see `load_top_50` function in `data_utilities.py`), but in the future we could extend this to other gene sets or all genes.

In [ ]:
def filter_cancer_types(gene, y_df, sample_freeze, mutation_burden):
    # most of this code is copied from process_y_matrix in pancancer_utilities.tcga_utilities
    # 
    # note this is not including copy number variants, to do that we have to
    # know oncogene/TSG status for every gene (need to figure out where to get
    # this info)
    y_df = pd.DataFrame(y_df)
    y_df.columns = ['status']
    y_df = (
        y_df.merge(
            sample_freeze, how='left', left_index=True, right_on='SAMPLE_BARCODE'
        )
        .set_index('SAMPLE_BARCODE')
        .merge(mutation_burden, left_index=True, right_index=True)
    )
    disease_counts_df = pd.DataFrame(y_df.groupby('DISEASE').sum()['status'])
    disease_proportion_df = disease_counts_df.divide(
        y_df['DISEASE'].value_counts(sort=False).sort_index(), axis=0
    )
    filter_disease_df = (disease_counts_df > cfg.filter_count) & (disease_proportion_df > cfg.filter_prop)
    disease_proportion_df['disease_included'] = filter_disease_df
    disease_proportion_df['count'] = disease_counts_df['status']
    filter_disease_df.columns = ['disease_included']
    return filter_disease_df, disease_proportion_df


def get_all_valid_combos():
    # not currently using this function, takes a while (5-10 minutes) to run
    valid_combos_df = pd.DataFrame()
    counter = 0
    for gene in overlap_genes:
        filter_df, _ = filter_cancer_types(gene, mutation_df.loc[:, gene],
                                           sample_freeze_df, mut_burden_df)
        valid_df = (
            filter_df.query('disease_included')
            .drop(['disease_included'], axis='columns')
            .reset_index()
            .rename({'DISEASE': 'disease'}, axis='columns')
        )
        valid_df['gene'] = gene
        if len(valid_df) > 0:
            valid_combos_df = pd.concat((valid_combos_df, valid_df))
        counter += 1
        if counter % 500 == 0:
            print('{} done'.format(counter), file=sys.stderr)
    print('done.', file=sys.stderr)
    return valid_combos_df

def get_top_valid_combos(top_genes_df):
    top_genes_combos_df = pd.DataFrame()
    for gene in top_genes_df['gene']:
        _, status_df = filter_cancer_types(gene, mutation_df.loc[:, gene],
                                           sample_freeze_df, mut_burden_df)
        status_df = status_df.reset_index()
        status_df['gene'] = gene
        status_df.rename({'DISEASE': 'disease'}, axis='columns', inplace=True)
        top_genes_combos_df = pd.concat((top_genes_combos_df, status_df))
    return top_genes_combos_df 

top_genes_df = du.load_top_50()
filtered_combos_df = get_top_valid_combos(top_genes_df)
filtered_combos_df.head()

In [ ]:
top_valid_df = (
    filtered_combos_df[filtered_combos_df.disease_included].drop(['disease_included'], axis='columns')
)
print(len(top_valid_df), 'combos out of', 50 *33, 'possibilities ({:.3f}%)'.format(len(top_valid_df) / (50 * 33)))
unique_genes = np.unique(top_valid_df.gene)
print(len(unique_genes), 'genes have valid combinations, out of', top_genes_df.shape[0], 'total')
unique_cancers = np.unique(top_valid_df.disease)
all_cancers = cancer_types_df['cancertype'].values
print(len(unique_cancers), 'cancers have valid combinations, out of', len(all_cancers), 'total')
print(unique_cancers)
print(set(all_cancers) - set(unique_cancers))

Now we plot the results, with different colors for diseases that are included and removed.

In [ ]:
filtered_combos_df['identifier'] = (
    filtered_combos_df['gene'] + '_' +
    filtered_combos_df['disease']
)

sns.set({'figure.figsize': (12, 8)})
sns.scatterplot(data=filtered_combos_df, x='status', y='count', hue='disease_included')
plt.title('Mutation trends for top 50 mutated genes in TCGA')
plt.xlabel('Proportion of samples mutated')
plt.ylabel('Number of samples mutated')

def label_points(x, y, val, ax):
    a = pd.DataFrame({'x': x, 'y': y, 'val': val})
    for i, point in a.iterrows():
        if point['x'] > 0.55 or point['y'] > 250:
            ax.text(point['x']+.01, point['y']+5, '(' + str(point['val']) + ')')
            
label_points(filtered_combos_df['status'], filtered_combos_df['count'], filtered_combos_df.identifier, plt.gca())

In [ ]:
cancer_type_to_plot = 'SKCM'
percent_cutoff = 0.05
plot_df = filtered_combos_df[
    (filtered_combos_df['disease'] == cancer_type_to_plot) &
    (filtered_combos_df['status'] > percent_cutoff)
].sort_values(by='status', ascending=False)

sns.set({'figure.figsize': (20, 4)})
sns.barplot(data=plot_df, x='gene', y='status', color='b')
plt.xlabel('Gene name')
plt.ylabel('Proportion of samples mutated in {}'.format(cancer_type_to_plot))

Takeaways here:
* There are more gene/cancer type combinations that **aren't** filtered out than we expected (we originally guessed that most combinations would be filtered out). Since we're filtering to only the top 50 most mutated genes this isn't that surprising in retrospect, but still good to know.
* Some of the highly mutated outliers from this plot make sense based on other studies/previous knowledge (e.g. TP53 in ovarian cancer, KRAS in pancreatic cancer, TTN in melanoma as a marker of high background mutation rate), so it's a useful sanity check.

Next, we'll compare some PCA plots of the data. This will help us to determine:
* Does the data separate by cancer type? (we would expect this)
* How many genes are necessary to preserve any signal? Does it matter if we select genes randomly, or using mean absolute deviation (high MAD = more information content?)

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=100)
X_proj = pca.fit_transform(rnaseq_df)
sns.set({'figure.figsize': (8, 6)})
sns.lineplot(range(len(pca.explained_variance_ratio_)), pca.explained_variance_ratio_)
plt.xlabel('PCA component')
plt.ylabel('Percent of variance explained')

In [ ]:
# make sure sample info order is the same as expression data order
assert sample_info_df.index.equals(rnaseq_df.index)

rnaseq_cancer_types = sample_info_df.cancer_type.values
# np.random.seed(3)
# unique_cancer_types = np.random.choice(np.unique(rnaseq_cancer_types), size=5)
# most common cancer types
unique_cancer_types = ['BRCA', 'KIRC', 'LUAD', 'THCA', 'UCEC', 'HNSC', 'LUSC']
sns.set({'figure.figsize': (10, 8)})

# TODO get colors from a colormap or something
for i, cancer_type in enumerate(unique_cancer_types):
    enum_samples_df = sample_info_df.reset_index()
    ixs = enum_samples_df.index[enum_samples_df['cancer_type'] == cancer_type].tolist()
    plt.scatter(X_proj[ixs, 0], X_proj[ixs, 1], label=cancer_type)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA projection of TCGA expression data, colored by cancer type (most common cancers)')
plt.legend()

In [ ]:
# subsample genes (either randomly or by MAD), and see if/when signal breaks down
def pca_subsampled_genes(genes, cancer_types, ax1, ax2, title1=None, title2=None):
    
    # subsample expression data by given genes/cancer types
    cancer_type_ixs = sample_info_df.index[sample_info_df['cancer_type'].isin(cancer_types)].tolist()
    subsampled_rnaseq_df = rnaseq_df.loc[cancer_type_ixs, genes]
    subsampled_samples_df = sample_info_df.loc[cancer_type_ixs, :].reset_index()
    
    # then do PCA to 2 dimensions
    num_components = min(100, subsampled_rnaseq_df.shape[1])
    pca = PCA(n_components=num_components)
    X_proj = pca.fit_transform(subsampled_rnaseq_df)
    
    # plot data points using the given axes
    for cancer_type in cancer_types:
        ixs = subsampled_samples_df.index[subsampled_samples_df['cancer_type'] == cancer_type].tolist()
        ax1.scatter(X_proj[ixs, 0], X_proj[ixs, 1], label=cancer_type)
    ax1.set_xlabel('PC1')
    ax1.set_ylabel('PC2')
    if title1 is not None:
        ax1.set_title(title1)
    else:
        ax1.set_title('PCA projection of subset of TCGA expression data, by cancer type')
    ax1.legend()
    
    # plot variance explained using the second axis
    sns.lineplot(range(len(pca.explained_variance_ratio_)), pca.explained_variance_ratio_, ax=ax2)
    ax2.set_xlabel('PCA component')
    ax2.set_ylabel('Percent of variance explained')
    if title2 is not None:
        ax2.set_title(title2)
    
sns.set({'figure.figsize': (16, 6)})
fig, axarr = plt.subplots(1, 2)
pca_subsampled_genes(rnaseq_df.columns, unique_cancer_types, axarr[0], axarr[1])
plt.show()

In [ ]:
subsample_nos = [10000, 5000, 1000, 100, 10]

sns.set({'figure.figsize': (25, 10)})
fig, axarr = plt.subplots(2, 5)
np.random.seed(2)
for ix, num_genes in enumerate(subsample_nos):
    ss_genes = np.random.choice(rnaseq_df.columns, size=num_genes)
    pca_subsampled_genes(ss_genes, unique_cancer_types, axarr[0, ix], axarr[1, ix],
                         title1='{} randomly sampled genes'.format(num_genes),
                         title2='Variance explained for {} genes'.format(num_genes))
plt.tight_layout()
plt.show()

In [ ]:
sns.set({'figure.figsize': (25, 10)})
fig, axarr = plt.subplots(2, 5)

mad_genes_df = (
    rnaseq_df.mad(axis=0)
             .sort_values(ascending=False)
             .reset_index()
)
mad_genes_df.columns = ['gene_id', 'mean_absolute_deviation']
sorted_genes = mad_genes_df.gene_id.astype(str).values

for ix, num_genes in enumerate(subsample_nos):
    ss_genes = sorted_genes[:num_genes]
    pca_subsampled_genes(ss_genes, unique_cancer_types, axarr[0, ix], axarr[1, ix],
                         title1='{} top genes by MAD'.format(num_genes),
                         title2='Variance explained for top {} MAD genes'.format(num_genes))
    
plt.tight_layout()
plt.show()

We can see that the data somewhat separates by cancer type, although there isn't a super strong signal. The data shape in the 2D PCA projection is fairly well preserved as genes are removed, particularly for mean absolute deviation (MAD) (the projection is almost indistinguishable until you get to 100 genes, or ~0.5% of the predictors in the dataset).

This will help us to inform how to subset the data during classification: it is likely that dropping genes with low MAD will have little/no effect on classification performance in most cases.